In [25]:
import time
import pandas as pd
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [26]:
DATA_DIR = Path.cwd().parent / 'data' / 'processed'
RANDOM_STATE = 25

In [27]:
X_train = pd.read_parquet(DATA_DIR / 'X_train.parquet')
y_train = pd.read_parquet(DATA_DIR / 'y_train.parquet')['Victims_Condition']

y_train_int, _ = pd.factorize(y_train, sort=True)
y_train_int = pd.Series(y_train_int, index=y_train.index)

In [28]:
numeric_cols     = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['category', 'bool']).columns.tolist()

In [29]:
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categorical_cols),
])

In [30]:
def make_pipe(clf):
    return Pipeline([('pre', preprocessor), ('clf', clf)])

In [31]:
models = {
    'LogisticRegression': make_pipe(LogisticRegression(max_iter=1000, n_jobs=-1, random_state=RANDOM_STATE)),
    'DecisionTree':       make_pipe(DecisionTreeClassifier(random_state=RANDOM_STATE)),
    'RandomForest':       make_pipe(RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=RANDOM_STATE)),
    'XGBoost':            make_pipe(XGBClassifier(tree_method='hist', n_jobs=-1, random_state=RANDOM_STATE, verbosity=0)),
    'LightGBM':           make_pipe(LGBMClassifier(n_jobs=-1, random_state=RANDOM_STATE, verbose=-1)),
    'CatBoost':           make_pipe(CatBoostClassifier(random_state=RANDOM_STATE, verbose=0)),
}

In [32]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = []

In [33]:
for name, model in models.items():
    print(f"{name}")
    t0 = time.time()

    y_for_cv = y_train_int if name == 'XGBoost' else y_train

    scores = cross_val_score(model, X_train, y_for_cv, cv=cv, scoring='f1_macro', n_jobs=-1)
    elapsed = time.time() - t0

    results.append({
        'model':           name,
        'cv_macro_f1':     scores.mean(),
        'cv_macro_f1_std': scores.std(),
        'time_sec':        round(elapsed, 1),
    })
    print(f"  mean = {scores.mean():.4f} (± {scores.std():.4f})  |  {elapsed:.1f}s")

cv_df = pd.DataFrame(results).sort_values('cv_macro_f1', ascending=False).reset_index(drop=True)
cv_df

LogisticRegression
  mean = 0.4044 (± 0.0009)  |  19.3s
DecisionTree
  mean = 0.4358 (± 0.0017)  |  201.2s
RandomForest
  mean = 0.4058 (± 0.0021)  |  1356.9s
XGBoost
  mean = 0.4156 (± 0.0021)  |  17.1s
LightGBM
  mean = 0.3987 (± 0.0021)  |  20.9s
CatBoost
  mean = 0.4184 (± 0.0023)  |  125.6s


,model,cv_macro_f1,cv_macro_f1_std,time_sec
0,DecisionTree,0.435800,0.001746,201.2
1,CatBoost,0.418408,0.002250,125.6
2,XGBoost,0.415605,0.002126,17.1
3,RandomForest,0.405770,0.002092,1356.9
4,LogisticRegression,0.404393,0.000932,19.3
5,LightGBM,0.398675,0.002092,20.9
